# DeBERTa-v3-base: проверка на пяти фолдах и финальное обучение

Дообучаем DeBERTa для прогноза оценки от 1 до 5. Обучаем **два верхних слоя энкодера, pooler и классификатор**; остальные веса заморожены (чтобы ускорить обучение, часть запусков выполнена на чужом более мощном Mac). Используем одну эпоху и максимум 128 токенов.

Сначала оцениваем фиксированные настройки на пяти групповых фолдах, затем независимо обучаем финальную модель на всех очищенных данных. Каждый запуск начинает с исходных предобученных весов. Результаты финальной модели на обучающей выборке не считаем оценкой качества.

Код обучения хранится в company_reviews/deberta_training.py. Ноутбук вызывает функции из этого файла; этот же код можно запустить из терминала без открытия Jupyter.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold

root = Path.cwd()
if not (root / "data" / "raw" / "train.csv").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
from company_reviews.baseline import normalize_reviews_for_split
from company_reviews.deberta_training import (
    check_checkpoint,
    checkpoint_path,
    probability_median,
    run,
    training_config,
)
from company_reviews.training import read_training_data

seed = 2026
torch.set_num_threads(4)
device = "mps" if torch.backends.mps.is_available() else "cpu"
data_path = root / "data" / "raw" / "train.csv"
cache_dir = root / "data" / "cache"

/Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Данные и разбиение

Удаляем точные дубликаты отзывов, оставляя первую строку. Проверяем отсутствие пропусков и допустимость оценок. Совпадающие тексты с разными оценками требуют проверки и останавливают запуск.

Группы строим без учёта регистра и повторяющихся пробелов. Одна группа целиком попадает либо в обучение, либо в проверку. Разбиение `StratifiedGroupKFold` с пятью фолдами и `random_state=2026` совпадает с разбиением в `baseline.ipynb`. Каждый отзыв проверяется ровно один раз.

In [3]:
clean = read_training_data(data_path)
groups = normalize_reviews_for_split(clean["Review"])
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
folds = list(splitter.split(clean["Review"], clean["Rating"], groups))

for train_idx, valid_idx in folds:
    assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[valid_idx])

valid_rows = np.concatenate([valid_idx for _, valid_idx in folds])
assert np.array_equal(np.sort(valid_rows), np.arange(len(clean)))
print(f"Отзывов после удаления дубликатов: {len(clean):,}")
print(f"Групп: {groups.nunique():,}; устройство: {device}")

Отзывов после удаления дубликатов: 59,976
Групп: 59,975; устройство: mps


In [4]:
fold_sizes = pd.DataFrame(
    [
        {"фолд": number, "обучение": len(train_idx), "проверка": len(valid_idx)}
        for number, (train_idx, valid_idx) in enumerate(folds, start=1)
    ]
)
display(fold_sizes.set_index("фолд"))

,обучение,проверка
фолд,,
1,47980,11996
2,47980,11996
3,47981,11995
4,47981,11995
5,47982,11994


## Зафиксированные настройки

Используем исходный SentencePiece-токенизатор и закреплённую ревизию `microsoft/deberta-v3-base`. Максимальная длина входа — 128 токенов, включая служебные. Более длинные отзывы обрезаются справа, короткие дополняются специальными токенами-заполнителями. Лимит выбран с учётом ограничений памяти и времени обучения, его оптимальность отдельно не проверялась. Обрезание может приводить к потере значимой информации в конце отзыва.

Во время обучения оценки 1–5 преобразуются в классы 0–4. Функция потерь — cross-entropy. Оптимизатор AdamW получает разные скорости для энкодера и головы; первые 5% шагов отведены прогреву, затем скорость линейно снижается. Градиенты ограничены нормой 1. Настройки фиксированы до пятифолдовой оценки.

Каждый фолд и финальное обучение начинаются с исходных предобученных весов, независимо от предыдущих запусков. Фиксируем seed для воспроизводимости; результаты могут немного различаться между устройствами и версиями библиотек.

In [5]:
configs = [
    training_config(data_path, clean, train_idx, valid_idx, device)
    for train_idx, valid_idx in folds
]
settings = configs[0]
parameter_names = [
    "model_id",
    "model_revision",
    "max_length",
    "batch_size",
    "num_epochs",
    "trainable_encoder_layers",
    "encoder_lr",
    "head_lr",
    "warmup_fraction",
    "max_grad_norm",
    "seed",
    "torch_version",
    "transformers_version",
]
display(pd.Series({name: settings[name] for name in parameter_names}, name="значение"))

model_id                                   microsoft/deberta-v3-base
model_revision              8ccc9b6f36199bec6961081d44eb72fb3f7353f3
max_length                                                       128
batch_size                                                         8
num_epochs                                                         1
trainable_encoder_layers                                           2
encoder_lr                                                   0.00003
head_lr                                                       0.0001
warmup_fraction                                                 0.05
max_grad_norm                                                    1.0
seed                                                            2026
torch_version                                                 2.14.1
transformers_version                                          4.57.6
Name: значение, dtype: object

## Обучение и проверка пяти фолдов

Последовательно обучаем модели на пяти фолдах и получаем вероятности классов для проверочных частей. Совместимые сохранённые результаты переиспользуются.

Если обучение остановится до завершения, при повторном запуске уже обученные фолды будут переиспользованы, а незавершённый — обучен заново.

In [6]:
progress_path = run(data_path, cache_dir, mode="cv", device=device)

Фолд 1: train=47980, valid=11996, /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_top2_5b9d7527ecb4


Сохранённая модель: /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_top2_5b9d7527ecb4


MAE фолда 1: 0.14721574


Фолд 2: train=47980, valid=11996, /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_top2_1c2ab390f018


Сохранённая модель: /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_top2_1c2ab390f018


...

Фолд 5: train=47982, valid=11994, /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_top2_e29130ffdfcb


Сохранённая модель: /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_top2_e29130ffdfcb


MAE фолда 5: 0.13873603


{
  "folds": [
    {
      "rows": 11996,
      "mae": 0.1472157385795265,
      "by_rating": {
        "1": {
          "rows": 3732,
          "mae": 0.06538049303322616
        },
        "2": {
          "rows": 326,
          "mae": 0.9233128834355828
        },
        "3": {
          "rows": 336,
          "mae": 1.1220238095238095
        },
        "4": {
          "rows": 670,
          "mae": 0.7686567164179104
        },
        "5": {
          "rows": 6932,
          "mae": 0.04746105020196192
        }
      },
      "checkpoint": "/Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_top2_5b9d7527ecb4",
      "weights_sha256": "94540ee6ab4d659a3cca89da28b525c6717ae5fd7b59e7153f9d49ae37cee0ac",
      "fold": 1
    },
    {
      "rows": 11996,
      "mae": 0.1464654884961654,
      "by_rating": {
        "1": {
          "rows": 3732,
          "mae": 0.05627009646302251
        },
        "2": {
          "rows": 326,
          "mae":

Результаты пяти фолдов: /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_cv_cf6d5fc3ad7b.json


## Качество

Из пяти вероятностей выбираем **медиану распределения**: первую оценку, для которой накопленная вероятность достигла 0,5. Такой прогноз минимизирует ожидаемую абсолютную ошибку для заданного распределения.

DeBERTa оценивается на пяти фолдах с одинаковыми настройками.

In [7]:
results = json.loads(progress_path.read_text(encoding="utf-8"))
assert results["complete"] and len(results["folds"]) == 5

scores = pd.DataFrame(
    [
        {"фолд": item["fold"], "строк": item["rows"], "MAE": item["mae"]}
        for item in results["folds"]
    ]
)
display(scores.set_index("фолд").round(5))
print(f"Средняя MAE: {results['mean_mae']:.5f}")
print(f"Стандартное отклонение: {results['std_mae']:.5f}")
print(f"MAE всех проверочных прогнозов: {results['oof_mae']:.5f}")

,строк,MAE
фолд,,
1,11996,0.14722
2,11996,0.14647
3,11995,0.13956
4,11995,0.14656
5,11994,0.13874


Средняя MAE: 0.14371
Стандартное отклонение: 0.00374
MAE всех проверочных прогнозов: 0.14371


In [8]:
oof_predictions = np.zeros(len(clean), dtype=np.int64)
for (_, valid_idx), config, result in zip(
    folds, configs, results["folds"], strict=True
):
    model_dir = checkpoint_path(cache_dir, config)
    weights_digest = result["weights_sha256"][:12]
    probability_path = model_dir / f"valid_probabilities_{weights_digest}.npy"
    probabilities = np.load(probability_path, allow_pickle=False)
    oof_predictions[valid_idx] = probability_median(probabilities, len(valid_idx))

labels = clean["Rating"].to_numpy()
absolute_errors = np.abs(labels - oof_predictions)
assert np.isclose(absolute_errors.mean(), results["oof_mae"])

In [9]:
by_rating = pd.DataFrame(
    [
        {
            "оценка": rating,
            "строк": int((labels == rating).sum()),
            "MAE": absolute_errors[labels == rating].mean(),
        }
        for rating in range(1, 6)
    ]
)
display(by_rating.set_index("оценка").round(5))

,строк,MAE
оценка,,
1,18660,0.05927
2,1629,0.95826
3,1679,1.14175
4,3349,0.76321
5,34659,0.04267


### Сравнение с baseline

| Модель | Средняя MAE по пяти фолдам |
| --- | ---: |
| TF-IDF по словам и символам + числовые признаки + Logistic Regression | 0,18507 |
| DeBERTa-v3-base | **0,14371** |

Для baseline приведён результат выбранных в `baseline.ipynb` настроек: `C=4`, `min_df=2`, вес числовых признаков `0,05`.

DeBERTa снижает среднюю MAE на **0,04137**, или **22,35%**, относительно baseline. При этом редкие оценки 2–4 остаются сложными. Низкая общая ошибка во многом определяется более точными прогнозами частых оценок 1 и 5.

Это результаты кросс-валидации, а не независимого итогового теста: параметры baseline подбирались на этих же пяти фолдах.

## Финальная модель на всех данных

После пятифолдовой проверки отдельно обучаем DeBERTa на всех 59 976 очищенных отзывах с теми же настройками. Сохраняем веса, токенизатор и конфигурацию для использования в API, инструкция подключения приведена в README.

In [10]:
run(data_path, cache_dir, mode="full", device=device)

full_config = training_config(
    data_path, clean, np.arange(len(clean)), np.array([], dtype=np.int64), device
)
final_model_dir = checkpoint_path(cache_dir, full_config)
assert check_checkpoint(final_model_dir, full_config)
print(f"Финальная модель: {final_model_dir}")
print(f"Обучающих строк: {full_config['train_rows']:,}")

Финальное обучение: 59976 строк, /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_full_33305736a2d2


Сохранённая модель: /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_full_33305736a2d2


Финальная модель: /Users/mikhail.shkarubsky/dev/vlanuka/company-reviews-deberta/data/cache/deberta_v3_base_full_33305736a2d2
Обучающих строк: 59,976


## Ограничения

В эксперименте использованы одна эпоха, максимум 128 токенов и обучение двух верхних слоёв энкодера, pooler и классификатора. Полное дообучение энкодера и альтернативные настройки не сравнивались. Длинные отзывы обрезаются, поэтому модель может терять значимую информацию в конце текста.

Групповое разбиение не допускает попадания отзывов, совпадающих после нормализации регистра и пробелов, одновременно в обучение и проверку. Семантически похожие, но текстуально различающиеся отзывы могут оказаться в разных фолдах.